# TODO 6.4：INT8 质量对比

对真实 SmoothQuant INT8 状态执行完整 FinQA dev 数值评测，并使用同一批样本对比 FP16/INT8 的长上下文数值正确性和 JSON 结构化输出成功率。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/baseline_quality.json'
BASELINE_REPORT = PROJECT_DIR / 'out/baseline_quality_experiment/baseline_quality_report.json'
INT8_MANIFEST = RESULT_DIR / 'smoothquant-int8-w8a8/quantized_model_manifest.json'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.finqa_metrics import evaluate_numeric_answers
from src.quantization.int8_fp8 import (
    check_structured_json, evaluate_int8_outputs,
    load_smoothquant_service, validate_int8_evaluation,
)
if not INT8_MANIFEST.is_file():
    raise RuntimeError('没有真实 INT8 manifest；请先运行 int8_fp8_experiment_colab.ipynb。')
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
baseline_metrics = json.loads(BASELINE_REPORT.read_text(encoding='utf-8'))['metrics']
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, RESULT_DIR / 'int8_quality_service.jsonl')
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_comparison_service.jsonl')

In [ ]:
predictions, references = [], []
batch_size = int(quality_config['batch_size'])
for start in range(0, len(assets.quality_dev), batch_size):
    rows = [assets.quality_dev[index] for index in range(start, min(start + batch_size, len(assets.quality_dev)))]
    requests = [InferenceRequest.from_finqa_row(row, request_id=f'int8-quality-{start + offset + 1:06d}') for offset, row in enumerate(rows)]
    results = int8_service.generate_batch(requests, prompt_suffix=quality_config.get('prompt_suffix', ''))
    predictions.extend(result['output_text'] for result in results)
    references.extend(str(row.get('answer') or row.get('qa', {}).get('answer')) for row in rows)
print('完整 dev 样本完成:', len(predictions))

In [ ]:
rows = [dict(row) for row in assets.quality_dev]
rows.sort(key=lambda row: len(InferenceRequest.from_finqa_row(row).render_prompt()), reverse=True)
long_results = []
for index, row in enumerate(rows[:3]):
    reference = str(row.get('answer') or row.get('qa', {}).get('answer'))
    request = InferenceRequest.from_finqa_row(row, request_id=f'long-{index + 1}')
    fp16_text = fp16_service.generate_one(request)['output_text']
    int8_text = int8_service.generate_one(request)['output_text']
    fp16_ok = evaluate_numeric_answers([fp16_text], [reference])['correct'] == 1
    int8_ok = evaluate_numeric_answers([int8_text], [reference])['correct'] == 1
    long_results.append({'index': index, 'input_characters': len(request.render_prompt()), 'baseline_passed': fp16_ok, 'passed': int8_ok})

structured_results = []
suffix = '\nReturn JSON with fields answer and unit only.'
for index, row in enumerate(rows[:10]):
    request = InferenceRequest.from_finqa_row(row, request_id=f'structured-{index + 1}')
    fp16_check = check_structured_json(fp16_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    int8_check = check_structured_json(int8_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    structured_results.append({'index': index, 'baseline_passed': fp16_check['passed'], **int8_check})

In [ ]:
payload = evaluate_int8_outputs(
    predictions, references, long_results, structured_results,
    tolerance=float(quality_config.get('tolerance', 1e-4)),
    baseline_numeric=baseline_metrics,
)
evaluation_path = RESULT_DIR / 'evaluation.json'
evaluation_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_int8_evaluation(evaluation_path)
validation = report.to_dict()
(RESULT_DIR / 'evaluation_validation.json').write_text(json.dumps(validation, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
print(json.dumps(validation, ensure_ascii=False, indent=2))
assert report.complete, 'INT8 三类质量对比未完整执行。'
print('TODO 6.4 数字、长上下文、结构化输出对比：PASS')
gc.collect()